# 09 – Pandas Practice

This is the capstone notebook for the pandas section.

You've learned:
- Series and DataFrames
- Selecting and filtering data
- Handling missing values
- Sorting and creating columns
- GroupBy and aggregation
- Merge and join
- Working with dates
- String operations

Now you put all of it together.

This notebook has **three projects** — each one is a realistic scenario that requires you to combine multiple skills. No step-by-step hints inside the problems. You decide what to do and in what order.

Attempt each project on your own first. The solutions are at the bottom of each project — but only check them after you've tried.

In [ ]:
import pandas as pd
import numpy as np

---
## Project 1 — Student Report Card

You are given three separate tables from a school's database. Your job is to produce a clean, sorted report card.

**The data:**

In [ ]:
students = pd.DataFrame({
    "Student_ID": [1, 2, 3, 4, 5, 6],
    "Name":       ["  amit sharma ", "PRIYA mehta", "ravi Kumar",
                   "neha SINGH", "karan Das", "SNEHA rao"],
    "DOB":        ["2005-04-12", "2005-08-23", "2006-01-05",
                   "2005-11-30", "2006-03-17", "2005-07-09"],
    "Dept_ID":    [1, 2, 1, 3, 2, 3]
})

marks = pd.DataFrame({
    "Student_ID": [1, 2, 3, 4, 5, 6],
    "Math":       [85, 92, np.nan, 78, 88, 95],
    "Science":    [78, 88, 82, np.nan, 91, 74],
    "English":    [90, 76, 85, 88, np.nan, 80]
})

departments = pd.DataFrame({
    "Dept_ID":    [1, 2, 3],
    "Department": ["Science", "Arts", "Commerce"]
})

**Your tasks:**

1. Clean the `Name` column — strip spaces and apply proper title case.
2. Convert `DOB` to datetime and calculate each student's current age in years.
3. Fill missing marks with the mean of that subject column.
4. Merge all three tables into one complete DataFrame.
5. Add a `Total` column (sum of all three subjects) and a `Percentage` column (Total out of 300).
6. Add a `Grade` column — A (>=85%), B (>=70%), C (>=55%), D (below 55%).
7. Drop `Dept_ID` and `DOB` from the final table.
8. Sort the final result by `Percentage` descending.
9. Print the final report.

In [ ]:
# Your solution here


<details>
<summary>Click to see solution</summary>

```python
# 1. Clean names
students["Name"] = students["Name"].str.strip().str.title()

# 2. Age calculation
students["DOB"] = pd.to_datetime(students["DOB"])
today = pd.Timestamp("today")
students["Age"] = ((today - students["DOB"]).dt.days / 365.25).astype(int)

# 3. Fill missing marks with column mean
for col in ["Math", "Science", "English"]:
    marks[col] = marks[col].fillna(marks[col].mean())

# 4. Merge all three tables
df = pd.merge(students, marks, on="Student_ID")
df = pd.merge(df, departments, on="Dept_ID")

# 5. Total and Percentage
df["Total"]      = df["Math"] + df["Science"] + df["English"]
df["Percentage"] = (df["Total"] / 300 * 100).round(1)

# 6. Grade
def assign_grade(p):
    if p >= 85:   return "A"
    elif p >= 70: return "B"
    elif p >= 55: return "C"
    else:         return "D"

df["Grade"] = df["Percentage"].apply(assign_grade)

# 7. Drop unnecessary columns
df = df.drop(columns=["Dept_ID", "DOB"])

# 8. Sort by Percentage
df = df.sort_values("Percentage", ascending=False).reset_index(drop=True)

# 9. Print
print(df)
```
</details>

---
## Project 2 — Sales Analysis

You have a raw sales dataset for a retail business. Analyse it and produce a summary report.

**The data:**

In [ ]:
sales = pd.DataFrame({
    "Order_ID":   [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    "Date":       ["2024-01-05", "2024-01-18", "2024-02-02", "2024-02-14",
                   "2024-03-09", "2024-03-22", "2024-04-01", "2024-04-15",
                   "2024-05-03", "2024-05-20"],
    "Salesperson":["Rahul", "Sneha", "Rahul", "Priya", "Sneha",
                   "Rahul", "Priya", "Sneha", "Priya", "Rahul"],
    "Region":     ["North", "South", "North", "West", "South",
                   "North", "West", "South", "West", "North"],
    "Product":    ["Laptop", "Phone", "Tablet", "Laptop", "Phone",
                   "Laptop", "Tablet", "Phone", "Laptop", "Tablet"],
    "Units":      [2, 5, 3, 1, 8, 2, 4, 6, 1, 3],
    "Unit_Price": [55000, 18000, 25000, 55000, 18000,
                   55000, 25000, 18000, 55000, 25000]
})

**Your tasks:**

1. Convert `Date` to datetime. Extract `Month` and `Quarter`.
2. Add a `Revenue` column — `Units × Unit_Price`.
3. Find the **total revenue per salesperson** — sorted highest to lowest.
4. Find the **total revenue per region**.
5. Find the **best selling product** (highest total units sold).
6. Find the **monthly revenue trend** — total revenue for each month.
7. Find which salesperson had the highest revenue in the **North** region only.
8. Add a `Performance` column — `"High"` if Revenue for that order is above the average order revenue, `"Low"` otherwise.

In [ ]:
# Your solution here


<details>
<summary>Click to see solution</summary>

```python
# 1. Date conversion
sales["Date"]    = pd.to_datetime(sales["Date"])
sales["Month"]   = sales["Date"].dt.month
sales["Quarter"] = sales["Date"].dt.quarter

# 2. Revenue
sales["Revenue"] = sales["Units"] * sales["Unit_Price"]

# 3. Revenue per salesperson
print("Revenue per Salesperson:")
print(
    sales.groupby("Salesperson")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

# 4. Revenue per region
print("\nRevenue per Region:")
print(sales.groupby("Region")["Revenue"].sum().reset_index())

# 5. Best selling product by units
print("\nBest Selling Product:")
print(
    sales.groupby("Product")["Units"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

# 6. Monthly revenue trend
print("\nMonthly Revenue:")
print(sales.groupby("Month")["Revenue"].sum().reset_index())

# 7. Top salesperson in North region
north = sales[sales["Region"] == "North"]
print("\nTop Salesperson in North:")
print(
    north.groupby("Salesperson")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .iloc[0]
)

# 8. Performance column
avg_revenue = sales["Revenue"].mean()
sales["Performance"] = np.where(sales["Revenue"] > avg_revenue, "High", "Low")
print("\nOrders with Performance tag:")
print(sales[["Order_ID", "Salesperson", "Revenue", "Performance"]])
```
</details>

---
## Project 3 — Employee Database Cleanup

HR has given you a messy employee export. Clean it, enrich it, and generate a department summary.

**The data:**

In [ ]:
employees = pd.DataFrame({
    "Emp_ID":     ["E001", "E002", "E003", "E004", "E005", "E006", "E007"],
    "Name":       ["  john doe ", "JANE SMITH", "bob martin",
                   "ALICE Brown", "charlie davis", "EVA green", "frank MILLER"],
    "Department": ["engineering", "Marketing", "ENGINEERING",
                   "marketing", "Sales", "Engineering", "SALES"],
    "Join_Date":  ["2019-03-15", "2020-07-01", "2018-11-20",
                   "2021-02-10", "2019-08-05", "2022-01-17", "2020-05-30"],
    "Salary":     [75000, 65000, np.nan, 70000, 55000, np.nan, 60000],
    "Email":      ["John@Company.COM", "jane@company.com", "BOB@company.com",
                   "alice@company.com", "CHARLIE@company.com",
                   "eva@company.com", "frank@company.com"]
})

**Your tasks:**

1. Clean `Name` — strip spaces, apply title case.
2. Clean `Department` — standardise to title case so `"engineering"`, `"ENGINEERING"`, `"Engineering"` all become `"Engineering"`.
3. Clean `Email` — lowercase everything.
4. Convert `Join_Date` to datetime. Calculate `Years_of_Experience` — how many full years since joining.
5. Fill missing `Salary` with the median salary of their department (not the overall median).
6. Add a `Seniority` column — `"Senior"` if experience >= 5 years, `"Mid"` if >= 3 years, `"Junior"` otherwise.
7. Extract `First_Name` from the `Name` column.
8. Build a department summary table — number of employees, average salary, average experience — sorted by average salary descending.
9. Print the cleaned employee table and the department summary.

In [ ]:
# Your solution here


<details>
<summary>Click to see solution</summary>

```python
# 1. Clean Name
employees["Name"] = employees["Name"].str.strip().str.title()

# 2. Clean Department
employees["Department"] = employees["Department"].str.strip().str.title()

# 3. Clean Email
employees["Email"] = employees["Email"].str.lower()

# 4. Join Date and Experience
employees["Join_Date"] = pd.to_datetime(employees["Join_Date"])
today = pd.Timestamp("today")
employees["Years_of_Experience"] = ((today - employees["Join_Date"]).dt.days / 365.25).astype(int)

# 5. Fill missing Salary with department median
employees["Salary"] = employees.groupby("Department")["Salary"].transform(
    lambda x: x.fillna(x.median())
)

# 6. Seniority
def seniority(yrs):
    if yrs >= 5:   return "Senior"
    elif yrs >= 3: return "Mid"
    else:          return "Junior"

employees["Seniority"] = employees["Years_of_Experience"].apply(seniority)

# 7. First Name
employees["First_Name"] = employees["Name"].str.split(" ").str[0]

# 8. Department summary
summary = employees.groupby("Department").agg(
    Headcount      = ("Emp_ID",             "count"),
    Avg_Salary     = ("Salary",             "mean"),
    Avg_Experience = ("Years_of_Experience","mean")
).reset_index()

summary["Avg_Salary"]     = summary["Avg_Salary"].round(0)
summary["Avg_Experience"] = summary["Avg_Experience"].round(1)
summary = summary.sort_values("Avg_Salary", ascending=False).reset_index(drop=True)

# 9. Print both
print("Cleaned Employee Table:")
print(employees.drop(columns=["Join_Date"]))
print()
print("Department Summary:")
print(summary)
```
</details>

---
## What's Next?

You've finished the entire pandas section. Here's what you covered:

| Notebook | Topic |
|---|---|
| 04_01 | Introduction to Pandas — Series |
| 04_02 | Selecting and Filtering Data |
| 04_03 | Handling Missing Values |
| 04_04 | Sorting and Creating Columns |
| 04_05 | GroupBy |
| 04_06 | Merge and Join |
| 04_07 | Working with Dates |
| 04_08 | String Operations |
| 04_09 | Practice (this notebook) |

**Up next: Section 05 — EDA and Visualization.**

You'll take everything you know about pandas and start turning data into charts and visual insights using Matplotlib and Seaborn.